# Lección 10.6 — Más allá de clasificar: detección, segmentación y Vision Transformers

> **Módulo 10 · Visión por computadora (CNN)** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 10.3–10.5, Lección 4.2 (precisión y *recall*), Lección 4.3 (umbrales)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir **clasificación, detección, segmentación semántica y de instancias**, y elegir la adecuada para un problema.
- Calcular la **IoU** (intersección sobre unión) y aplicar **NMS** para limpiar las cajas de un detector.
- Medir una segmentación con **IoU** y **Dice**, y explicar la idea de la U-Net.
- Ver **dónde mira** una red de clasificación (mapas de activación) y explicar cómo un **Vision Transformer** convierte una imagen en una secuencia de parches.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> 🖥️ Todo corre en CPU en pocos minutos (también en Google Colab sin GPU). Si tienes GPU, se usará sola.

## ▶️ Preparación
Usaremos la ResNet18 preentrenada de la Lección 10.5 (≈45 MB; la primera vez se descarga) y la foto del templo de la Lección 10.1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
from torchvision import models
from torchvision.ops import box_iou, nms
from sklearn.datasets import load_sample_image

plt.rcParams["figure.figsize"] = (8, 4.5)
foto = load_sample_image("china.jpg")                 # (427, 640, 3)
print("✅ Todo listo")

---
## 1. 📖 La intuición: describir una foto con cada vez más detalle

Imagina que le describes una foto a alguien por teléfono:

| Nivel de detalle | Lo que dices | Tarea de visión | Salida del modelo |
|---|---|---|---|
| 1 | "Hay un perro" | **Clasificación** | Una etiqueta (+ confianza) |
| 2 | "Hay un perro, abajo a la izquierda" | **Localización / detección** | Cajas (*bounding boxes*) con etiqueta y confianza, **una por objeto** |
| 3 | "Estos píxeles son perro, estos son pasto, estos son cielo" | **Segmentación semántica** | Una etiqueta **por píxel** |
| 4 | "Estos píxeles son el perro 1 y estos el perro 2" | **Segmentación de instancias** | Una máscara por **cada objeto** |

```
Clasificación        Detección              Segmentación semántica
┌───────────┐        ┌───────────┐          ┌───────────┐
│   🐕      │        │ ┌──┐      │          │░░░░░░░░░░░│
│      🐕   │  →     │ │🐕│ ┌──┐ │    →     │░▓▓░░░▓▓░░░│  ▓ = perro
│           │        │ └──┘ │🐕│ │          │░▓▓░░░▓▓░░░│  ░ = fondo
└───────────┘        └──────┴──┴─┘          └───────────┘
 "perro"              2 cajas "perro"         cada píxel con su clase
```

### ✋ Ejercicio 1 — ¿Qué tarea es?
Para cada aplicación, di qué tarea necesita (clasificación, detección, segmentación semántica o de instancias):
1. Una app que dice si una foto de una hoja tiene una plaga o no.
2. Un auto autónomo que necesita saber dónde está cada peatón.
3. El "modo retrato" del celular, que desenfoca todo **menos** a la persona.
4. Contar **cuántas** células hay en una imagen de microscopio, aunque estén pegadas unas con otras, y medir el área de cada una.
5. Un sistema que cuenta autos en un estacionamiento.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Clasificación** (sí/no por foto).
2. **Detección** (una caja por peatón, en tiempo real). Muchos autos usan además segmentación para saber qué píxeles son carretera.
3. **Segmentación** semántica (persona vs. fondo, píxel a píxel): una caja no basta para recortar el pelo.
4. **Segmentación de instancias**: necesitas separar cada célula (instancia) y su forma exacta para medir el área.
5. **Detección** (contar cajas). Si los autos están muy apretados, segmentación de instancias.
</details>

---
## 2. 📖 Detección de objetos: cajas, IoU y NMS

Un detector produce, para cada objeto: una **caja** `(x1, y1, x2, y2)` (esquina superior izquierda e inferior derecha, en píxeles), una **clase** y una **confianza**.

Para saber si una caja predicha es buena, la comparamos con la caja correcta (la que dibujó una persona) usando la **IoU** (*Intersection over Union*, intersección sobre unión):

$$\text{IoU} = \frac{\text{área donde se solapan}}{\text{área total que cubren entre las dos}}$$

- IoU = 1 → cajas idénticas · IoU = 0 → no se tocan.
- Por convención, una detección se cuenta como **correcta** si su IoU con la caja real es **≥ 0.5**.

In [ ]:
def iou(a, b):
    """IoU entre dos cajas (x1, y1, x2, y2)."""
    ancho = max(0, min(a[2], b[2]) - max(a[0], b[0]))       # ancho de la zona común
    alto = max(0, min(a[3], b[3]) - max(a[1], b[1]))        # alto de la zona común
    interseccion = ancho * alto
    area = lambda c: (c[2] - c[0]) * (c[3] - c[1])
    return interseccion / (area(a) + area(b) - interseccion)

def dibujar_cajas(cajas, colores, textos, img=foto):
    fig, ax = plt.subplots(figsize=(8, 5.5)); ax.imshow(img); ax.axis("off")
    for c, col, t in zip(cajas, colores, textos):
        ax.add_patch(patches.Rectangle((c[0], c[1]), c[2] - c[0], c[3] - c[1], fill=False, color=col, lw=2.5))
        ax.text(c[0], c[1] - 5, t, color="white", fontsize=9, backgroundcolor=col)
    plt.show()

real = (120, 30, 360, 400)          # la caja "verdadera" del templo, dibujada por una persona
pred = (150, 60, 390, 420)         # ✏️ la caja que predijo un detector: mueve sus números
dibujar_cajas([real, pred], ["green", "red"], ["real", f"predicha · IoU = {iou(real, pred):.2f}"])

### 🔮 Ejercicio 2 — Calcula la IoU a mano
Dos cajas de **10 × 10**: `A = (0, 0, 10, 10)` y `B = (5, 0, 15, 10)` (B está movida 5 píxeles a la derecha).
1. ¿Cuál es el área de la intersección? ¿Y de la unión?
2. ¿Cuál es la IoU? ¿Contaría como detección correcta?
3. ¿Y si B estuviera movida 5 a la derecha **y** 5 hacia abajo: `(5, 5, 15, 15)`?

✍️ *Mis cuentas:*

In [ ]:
A, B1, B2 = (0, 0, 10, 10), (5, 0, 15, 10), (5, 5, 15, 15)
print(f"IoU(A, B1) = {iou(A, B1):.3f}")
print(f"IoU(A, B2) = {iou(A, B2):.3f}")
# torchvision trae la misma función, para muchas cajas a la vez:
print(box_iou(torch.tensor([A], dtype=torch.float), torch.tensor([B1, B2], dtype=torch.float)))

<details><summary><b>👀 Ver respuesta</b></summary>

1. Intersección: 5 × 10 = **50**. Unión: 100 + 100 − 50 = **150**.
2. IoU = 50 / 150 = **0.333**. **No** cuenta como correcta (< 0.5), aunque "la mitad de la caja está bien".
3. Intersección 5 × 5 = 25; unión 200 − 25 = 175; IoU = **0.143**. Moverse en dos direcciones castiga mucho.

La IoU es exigente: con cajas pequeñas, unos pocos píxeles de error bastan para fallar.
</details>

### 📖 NMS: limpiar cajas repetidas
Un detector no produce una caja por objeto: produce **cientos** de cajas candidatas, muchas casi encima de otras sobre el mismo objeto. La **supresión de no máximos** (*Non-Maximum Suppression*, NMS) las limpia:

1. Ordena las cajas por confianza.
2. Quédate con la de **mayor** confianza.
3. **Elimina** todas las que se solapan con ella con IoU > umbral (p. ej. 0.5): "son el mismo objeto".
4. Repite con las que quedan.

Es como un concurso donde, entre los que dicen lo mismo, solo habla el más seguro.

In [ ]:
cajas = np.array([
    [120,  30, 360, 400],    # 0: templo
    [130,  40, 370, 410],    # 1: templo, casi igual que la 0
    [110,  20, 340, 390],    # 2: templo, otra variante
    [520, 150, 640, 420],    # 3: árboles de la derecha
    [530, 170, 640, 410],    # 4: árboles, variante
    [400,  60, 500, 140],    # 5: ¿una nube? (en el cielo)
])
confianza = np.array([0.92, 0.85, 0.80, 0.70, 0.75, 0.40])

dibujar_cajas(cajas, ["red"] * 6, [f"{i}: {c:.2f}" for i, c in enumerate(confianza)])

### 🔮 Ejercicio 3 — Juega a ser NMS
Con umbral de IoU = 0.5, ¿qué cajas **sobreviven**? Razónalo con el algoritmo (las cajas 0, 1 y 2 se solapan mucho; la 3 y la 4 también).

✍️ *Cajas que sobreviven:*

In [ ]:
def nms_a_mano(cajas, confianza, umbral=0.5):
    orden = list(np.argsort(-confianza))      # de mayor a menor confianza
    quedan = []
    while orden:
        mejor = orden.pop(0)
        quedan.append(int(mejor))
        orden = [i for i in orden if iou(cajas[mejor], cajas[i]) <= umbral]   # elimina las repetidas
    return quedan

UMBRAL_NMS = 0.5     # ✏️ prueba 0.3 y 0.9
print("NMS a mano   :", nms_a_mano(cajas, confianza, UMBRAL_NMS))
print("torchvision  :", nms(torch.tensor(cajas, dtype=torch.float), torch.tensor(confianza, dtype=torch.float), UMBRAL_NMS).tolist())

<details><summary><b>👀 Ver respuesta</b></summary>

Sobreviven **0, 4 y 5**:
- La **0** (0.92) gana entre las del templo y elimina a la 1 y la 2 (IoU > 0.5 con ella).
- Entre las cajas de los árboles gana la **4** (0.75), no la 3 (0.70): manda la confianza, no el orden de la lista.
- La **5** no se solapa con nadie y sobrevive, aunque su confianza sea baja (0.40). Para eliminarla hace falta otro filtro: un **umbral de confianza** (Sección 5).

✏️ Con `UMBRAL_NMS = 0.9` ninguna pareja se considera repetida (sus IoU rondan 0.8) y sobreviven **las 6**: cajas duplicadas. Con 0.3 aquí no cambia nada, porque las cajas de objetos distintos no se tocan; pero en general, cuanto más bajo el umbral, más cajas se eliminan. Si dos objetos reales están muy juntos (dos personas abrazadas), un umbral bajo puede borrar uno de ellos.
</details>

### 📖 Cómo funcionan los detectores (sin entrar en detalles)
Todos usan una CNN (como ResNet) como "extractor de características" y le añaden una **cabeza** que, para muchas posiciones del mapa, predice "¿hay objeto aquí? ¿de qué clase? ¿cómo ajusto la caja?". Luego aplican NMS.

| Familia | Ejemplos | Idea | Punto fuerte |
|---|---|---|---|
| Dos etapas | Faster R-CNN, Mask R-CNN | 1) propone regiones candidatas, 2) clasifica y ajusta cada una | Precisión |
| Una etapa | **YOLO**, SSD, RetinaNet | Predice cajas y clases de una sola pasada sobre una cuadrícula | **Velocidad** (tiempo real) |
| Transformers | DETR | Predice directamente un conjunto de objetos, sin NMS | Diseño simple |

> 🧪 **Para Colab (opcional, descarga ≈75 MB):** `torchvision` trae detectores preentrenados en COCO (80 clases: personas, autos, perros…).
> ```python
> from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
> pesos = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
> detector = fasterrcnn_mobilenet_v3_large_fpn(weights=pesos).eval()
> img = torch.tensor(foto).permute(2, 0, 1).float() / 255
> salida = detector([img])[0]                       # dict con 'boxes', 'labels', 'scores' (¡ya con NMS!)
> for caja, etiqueta, conf in zip(salida["boxes"], salida["labels"], salida["scores"]):
>     if conf > 0.5:
>         print(pesos.meta["categories"][etiqueta], caja.int().tolist(), f"{conf:.2f}")
> ```

---
## 3. 📖 Segmentación: una etiqueta por píxel

En segmentación, la "respuesta" es otra imagen del mismo tamaño, la **máscara**, donde cada píxel dice a qué clase pertenece. Para medir qué tan buena es una máscara predicha se usan:
- **IoU** (la misma idea que con cajas, pero contando píxeles): píxeles en ambas / píxeles en alguna.
- **Dice** = 2 × (píxeles en ambas) / (píxeles de la real + píxeles de la predicha). Es muy usado en medicina. Siempre es ≥ IoU.

Simulemos una máscara real (un círculo, como un tumor en una resonancia) y una predicha algo desplazada:

In [ ]:
yy, xx = np.mgrid[0:100, 0:100]
mascara_real = (xx - 50) ** 2 + (yy - 50) ** 2 < 20 ** 2           # círculo de radio 20 en el centro

DESPLAZAMIENTO = 6     # ✏️ cuántos píxeles se equivoca la predicción
RADIO_PRED = 20        # ✏️ ¿y si predice un círculo más grande?
mascara_pred = (xx - 50 - DESPLAZAMIENTO) ** 2 + (yy - 50) ** 2 < RADIO_PRED ** 2

ambas = (mascara_real & mascara_pred).sum()
alguna = (mascara_real | mascara_pred).sum()
iou_mask = ambas / alguna
dice = 2 * ambas / (mascara_real.sum() + mascara_pred.sum())

vista = np.zeros((100, 100, 3)); vista[mascara_real] += [0, 0.8, 0]; vista[mascara_pred] += [0.8, 0, 0]
plt.figure(figsize=(4, 4)); plt.imshow(vista.clip(0, 1)); plt.axis("off")
plt.title(f"verde = real · rojo = predicha · amarillo = ambas\nIoU = {iou_mask:.2f} · Dice = {dice:.2f}", fontsize=9)
plt.show()

### 🔮 Ejercicio 4 — El círculo gordo
Un modelo "tramposo" predice un círculo **enorme** (radio 40) centrado en el lugar correcto, que cubre todo el tumor y mucho más. Antes de probarlo (`DESPLAZAMIENTO = 0`, `RADIO_PRED = 40`):
1. ¿Qué porcentaje del tumor real queda cubierto?
2. ¿La IoU será alta o baja?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El **100 %** del tumor queda cubierto (el *recall* de píxeles es perfecto).
2. Pero la IoU es **baja**: ≈0.25 (Dice ≈0.40), porque la unión es el círculo grande entero (4 veces el área del real). La IoU castiga por igual "lo que te falta" y "lo que te sobra", así que un modelo no puede hacer trampa pintando de más. Con `DESPLAZAMIENTO = 6` y radio correcto, en cambio, la IoU es ≈0.68 y el Dice ≈0.81.
</details>

### 📖 La U-Net: bajar para entender, subir para dibujar
Una CNN de clasificación **reduce** la imagen (pooling) para entender *qué* hay, pero pierde el *dónde*. Para segmentar hay que volver al tamaño original. La **U-Net** (2015, creada para imágenes médicas) lo hace en forma de U:

```
Entrada 128×128 ─► conv ─────────────── atajo (copia) ───────────────► conv ─► Máscara 128×128
                    │ pool                                              ▲ subir ×2
                    ▼                                                   │
                   64×64 ─► conv ───────── atajo ──────────► conv ──────┘
                             │ pool                           ▲ subir ×2
                             ▼                                │
                            32×32 ─────► conv (el "fondo") ───┘
         ENCODER: entiende QUÉ hay                     DECODER: recupera DÓNDE está
```

- La mitad izquierda (**encoder**) es una CNN normal.
- La mitad derecha (**decoder**) **agranda** los mapas con convoluciones transpuestas ("deconvoluciones").
- Los **atajos** copian los mapas detallados del encoder al decoder para que los bordes salgan finos (¡otra vez atajos, como en ResNet!).

In [ ]:
encoder = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
subir = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)     # "deconvolución": duplica alto y ancho

x = torch.rand(1, 3, 128, 128)
print("Entrada              :", tuple(x.shape))
print("Tras el encoder      :", tuple(encoder(x).shape))
print("Tras una subida (×2) :", tuple(subir(encoder(x)).shape))

### ✋ Ejercicio 5
1. ¿Cuántas capas `subir` como esta harían falta para volver de 32 × 32 a 128 × 128?
2. La última capa de una U-Net para segmentar **3 clases** (fondo, hígado, tumor) debe producir un tensor de forma `(1, ?, 128, 128)`. ¿Qué número va en el `?`

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Dos**: 32 → 64 → 128.
2. **3**: un "mapa de puntuaciones" por clase, para cada píxel. La clase predicha de cada píxel es la de mayor puntuación (`argmax` en la dimensión de canales). Es exactamente una clasificación… repetida en cada uno de los 16,384 píxeles.
</details>

---
## 4. 📖 ¿Dónde mira la red? Mapas de activación de clase (CAM)

Una ResNet de clasificación **no** fue entrenada para localizar, pero aun así "sabe" dónde está el objeto. Recuerda su final (Lección 10.4):

```
layer4 → 512 mapas de 7×7 → pooling global → 512 números → fc → 1,000 clases
```

Cada clase tiene 512 pesos en `fc`: cuánto le importa cada mapa. Si sumamos los 512 mapas de 7 × 7 **ponderados por los pesos de la clase ganadora**, obtenemos un mapa de calor de 7 × 7 que dice *qué zonas empujaron la decisión*. Es el **CAM** (*Class Activation Map*, Zhou et al., 2016).

In [ ]:
pesos = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=pesos).eval()
preparar = pesos.transforms()                          # redimensiona a 256, recorta 224 y normaliza (Lección 10.5)
categorias = pesos.meta["categories"]

def cam(img_np):
    x = preparar(torch.tensor(img_np).permute(2, 0, 1)).unsqueeze(0)
    extractor = nn.Sequential(*list(resnet.children())[:-2])          # todo menos avgpool y fc
    with torch.no_grad():
        mapas = extractor(x)[0]                                        # (512, 7, 7)
        probs = resnet(x).softmax(1)[0]
    clase = probs.argmax().item()
    calor = torch.einsum("c,chw->hw", resnet.fc.weight[clase].detach(), mapas)   # suma ponderada de los 512 mapas
    return x, calor, clase, probs[clase].item()

x, calor, clase, conf = cam(foto)
img_vista = (x[0].permute(1, 2, 0) * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])).clamp(0, 1)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
ax[0].imshow(img_vista); ax[0].set_title("Lo que ve la red (224×224)")
ax[1].imshow(calor, cmap="jet"); ax[1].set_title("CAM (7×7)")
ax[2].imshow(img_vista); ax[2].imshow(calor, cmap="jet", alpha=0.45, extent=(0, 224, 224, 0), interpolation="bilinear")
ax[2].set_title(f"Predicción: '{categorias[clase]}' ({conf:.0%})")
for a in ax: a.axis("off")
plt.show()

### ✋ Ejercicio 6 — Interpreta el mapa
1. ¿Qué clase predice la red? (Busca su traducción si no la conoces.) ¿Tiene sentido?
2. ¿Qué zona de la foto se ilumina más? ¿Es la que tú mirarías para decidir esa clase?
3. ¿Por qué el mapa original es de solo 7 × 7?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Predice **'bell cote'** (espadaña o campanario pequeño en el tejado de un edificio) con ≈53 %, y su segunda opción es 'palace' (palacio). Para un templo chino con tejados ornamentados es una respuesta razonable: ImageNet no tiene la clase "templo chino".
2. Se ilumina sobre todo **la pagoda**: sus tejados apilados y el cuerpo del edificio, justo lo que distingue a esa clase. El cielo, el lago y la vegetación casi no aportan.
3. Porque es el tamaño de los mapas de `layer4`: la imagen de 224 se dividió entre 32 (224 / 32 = 7). Por eso el CAM es "borroso": cada casilla corresponde a un bloque de 32 × 32 píxeles.

El CAM es una herramienta de **explicabilidad** (Lección 15.6): sirve para detectar si la red decide por las razones correctas (p. ej., que un detector de "lobo" no esté mirando la nieve del fondo).
</details>

---
## 5. 📖 Hiperparámetros de detección y segmentación

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Umbral de **confianza** del detector | Qué cajas se muestran | Menos cajas: más precisión, menos *recall* (se escapan objetos) | Más cajas: más *recall*, más falsas alarmas | 0.5 (ajustar según el costo de cada error, Lección 4.8) |
| Umbral de **IoU de NMS** | Cuándo dos cajas son "el mismo objeto" | Elimina menos: quedan duplicados | Elimina más: puede borrar objetos pegados | 0.5 (0.45–0.7) |
| Umbral de **IoU para "acierto"** al evaluar | Qué tan exacta debe ser una caja | Evaluación más estricta | Más permisiva | 0.5 (COCO promedia de 0.5 a 0.95) |
| **Resolución de entrada** | Tamaño de objeto más pequeño detectable | Detecta objetos pequeños; más lento | Más rápido; pierde objetos pequeños | 640 px (YOLO), 800 px (Faster R-CNN) |
| **Tamaño de parche** (ViT, Sección 6) | Detalle de cada "palabra visual" | Menos parches: más rápido, menos detalle | Más parches: más detalle, mucho más costo | 16 × 16 |

### 🔮 Ejercicio 7 — El umbral de confianza
En la escena del ejercicio 3, después de NMS quedaron las cajas 0 (0.92), 4 (0.75) y 5 (0.40). Supón que **solo** el templo y los árboles son objetos reales (la caja 5 está en el cielo vacío). ¿Qué pasa con la precisión y el *recall* (Lección 4.2) si el umbral de confianza es 0.3? ¿Y si es 0.8?

<details><summary><b>👀 Ver respuesta</b></summary>

- **Umbral 0.3**: se muestran las 3 cajas → 2 aciertos y 1 falsa alarma → precisión 2/3 ≈ 0.67, *recall* 2/2 = 1.
- **Umbral 0.8**: solo se muestra la del templo → precisión 1/1 = 1, *recall* 1/2 = 0.5 (se nos escaparon los árboles).

Es el mismo compromiso que con los umbrales de la Lección 4.3. Las métricas de detección como el **mAP** (*mean Average Precision*) resumen la curva precisión–*recall* de todos los umbrales; lo practicarás en los ejercicios finales.
</details>

---
## 6. 📖 Vision Transformers: una imagen como una frase de parches

En 2020, Google mostró que se podía clasificar imágenes **sin convoluciones**, con la arquitectura de los modelos de lenguaje (el Transformer, Módulo 12). La idea del **ViT** (*Vision Transformer*): *"una imagen vale 16 × 16 palabras"*.

1. Corta la imagen en **parches** de 16 × 16 píxeles (como las piezas de un rompecabezas).
2. Convierte cada parche en un vector (un "*token*", como una palabra).
3. Pasa la secuencia de parches por un Transformer, donde cada parche puede "prestar atención" a **todos** los demás desde la primera capa (Lección 12.1).

```
Imagen 224×224  →  14 × 14 = 196 parches de 16×16  →  196 "palabras"  →  Transformer  →  "templo"
```

In [ ]:
recorte = img_vista.numpy()                  # la foto de 224×224 que vio la ResNet
P = 16                                       # ✏️ tamaño del parche: prueba 32 o 56
n = 224 // P
fig, ax = plt.subplots(n, n, figsize=(6, 6))
for i in range(n):
    for j in range(n):
        ax[i, j].imshow(recorte[i * P:(i + 1) * P, j * P:(j + 1) * P]); ax[i, j].axis("off")
plt.suptitle(f"{n} × {n} = {n * n} parches de {P}×{P}", y=0.92); plt.show()

### ✋ Ejercicio 8 — Cuenta los parches
1. Imagen de 224 × 224, parches de 16: ¿cuántos parches? ¿Y con parches de 32?
2. Imagen de **384 × 384** con parches de 16: ¿cuántos parches?
3. En un Transformer, cada parche se compara con **todos** los demás. Si el número de parches se duplica, ¿cuántas comparaciones más hay?

✍️ *Mis cuentas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. (224 / 16)² = 14² = **196** parches. Con parches de 32: 7² = **49**.
2. (384 / 16)² = 24² = **576** parches.
3. Las comparaciones son "todos contra todos": n × n. Si n se duplica, hay **4 veces** más. Por eso los ViT se encarecen muy rápido con la resolución, y por eso se usan parches y no píxeles sueltos (¡224 × 224 = 50,176 píxeles serían 2,500 millones de comparaciones por capa!).
</details>

### 📖 El parche es… una convolución
Convertir cada parche de 16 × 16 × 3 en un vector se hace con una **convolución** de kernel 16 y *stride* 16 (¡los filtros no se solapan!). Así empieza, literalmente, el código de un ViT:

In [ ]:
a_parches = nn.Conv2d(3, 768, kernel_size=16, stride=16)     # 768 = tamaño de cada "palabra" en ViT-Base
x = torch.rand(1, 3, 224, 224)
mapa = a_parches(x)
print("Tras la conv de parches :", tuple(mapa.shape), "→ un vector de 768 por cada casilla de 14×14")
secuencia = mapa.flatten(2).transpose(1, 2)
print("Como secuencia de tokens:", tuple(secuencia.shape), "→ 196 'palabras' de 768 números")

### 📖 CNN vs. Vision Transformer

| | CNN (ResNet) | Vision Transformer (ViT) |
|---|---|---|
| **Pieza básica** | Convolución local (lupa de 3 × 3) | Atención entre todos los parches |
| **Qué "sabe" de entrada** | Que los vecinos importan y que un patrón puede estar en cualquier sitio (**sesgo inductivo** fuerte) | Casi nada: lo tiene que aprender todo de los datos |
| **Campo receptivo** | Crece capa a capa (Lección 10.3) | Global desde la primera capa |
| **Datos que necesita** | Funciona bien con datasets medianos | Brilla con datasets **enormes** (decenas o cientos de millones de imágenes) o preentrenado |
| **Costo con la resolución** | Crece con los píxeles | Crece con el **cuadrado** del número de parches |
| **Hoy** | Estándar en celulares, tiempo real y datos modestos | Estado del arte en tareas grandes, modelos multimodales (CLIP, modelos de lenguaje con visión) |

> 💡 La frontera se ha difuminado: hay CNN modernas inspiradas en los Transformers (ConvNeXt) y Transformers con ideas de las CNN (Swin, que usa ventanas locales). Y los modelos que "ven" y "hablan" a la vez (como los LLMs multimodales, Módulo 13) suelen usar un ViT como "ojos".

### 🔮 Ejercicio 9 — ¿CNN o ViT?
Tienes **5,000** radiografías etiquetadas y ningún modelo preentrenado. ¿Esperas que funcione mejor una CNN o un ViT entrenados desde cero? ¿Por qué? ¿Cambiaría tu respuesta si tuvieras un ViT preentrenado en cientos de millones de imágenes?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Desde cero y con 5,000 imágenes, una **CNN**: su "sesgo inductivo" (vecindarios, pesos compartidos) es conocimiento previo que el ViT tendría que aprender de los datos, y 5,000 imágenes no le alcanzan (el artículo original de ViT mostró que pierde contra ResNet con datasets medianos).

Con un ViT **preentrenado** a gran escala, la respuesta cambia: el *transfer learning* (Lección 10.5) le da lo que le faltaba, y un ViT preentrenado ajustado suele igualar o superar a una CNN. En la práctica: prueba ambos con validación y decide por métrica, velocidad y tamaño.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Varias cajas sobre el mismo objeto | Falta NMS o el umbral de NMS es muy alto | Aplicar NMS con IoU ≈ 0.5 |
| Dos objetos pegados y el detector solo ve uno | Umbral de NMS demasiado bajo | Subir el umbral de NMS |
| Se escapan los objetos pequeños | Resolución de entrada baja o demasiado *pooling* | Más resolución; detectores con mapas multiescala |
| Segmentación con "buena exactitud" de píxeles pero inútil | El fondo domina: predecir todo como fondo ya da 95 % de píxeles correctos | Medir con **IoU/Dice** por clase, no con exactitud |
| La red clasifica bien pero por la razón equivocada (mira el fondo) | Atajos espurios en los datos (todas las fotos de lobos tenían nieve) | Revisar con CAM; diversificar los datos |
| Cajas en el formato equivocado | Unas librerías usan `(x1, y1, x2, y2)` y otras `(x, y, ancho, alto)` o `(centro_x, centro_y, ancho, alto)` | Revisar la documentación y convertir (`torchvision.ops.box_convert`) |

### 🔮 Ejercicio 10 — La trampa de la exactitud en segmentación
En una resonancia de 100 × 100 píxeles, el tumor ocupa **400** píxeles. Un modelo pésimo predice **todo** como "sano". ¿Qué exactitud de píxeles tiene? ¿Y qué IoU para la clase tumor?

In [ ]:
total, tumor = 100 * 100, 400
print(f"Exactitud de píxeles del modelo 'todo sano': {(total - tumor) / total:.0%}")
print(f"IoU de la clase tumor: {0 / tumor:.0%}   (no acertó ni un píxel de tumor)")

<details><summary><b>👀 Ver respuesta</b></summary>

Exactitud de píxeles del **96 %**… y una IoU del tumor de **0 %**. Es el problema de los datos desbalanceados de la Lección 2.7, a nivel de píxel. Por eso la segmentación se evalúa con IoU o Dice **por clase** (y su promedio, el *mIoU*).
</details>

---
## 8. 📋 Ficha: tareas de visión más allá de clasificar

| Tarea | Salida | Modelos típicos | Métrica típica |
|---|---|---|---|
| **Clasificación** | Una etiqueta por imagen | ResNet, EfficientNet, ViT | Exactitud, top-5 |
| **Detección** | Cajas + clase + confianza | YOLO, Faster R-CNN, DETR | **mAP** (a IoU 0.5 o 0.5–0.95) |
| **Segmentación semántica** | Una clase por píxel | U-Net, DeepLab | **IoU/mIoU, Dice** |
| **Segmentación de instancias** | Una máscara por objeto | Mask R-CNN, SAM (*Segment Anything*) | mAP de máscaras |
| **Explicación (CAM)** | Mapa de calor de "dónde mira" | Cualquier CNN con pooling global | Revisión humana |

## 9. 🧠 Resumen en 5 puntos
1. Más allá de "¿qué hay?" están "¿**dónde** está?" (detección, con cajas) y "¿**qué píxeles** son?" (segmentación, con máscaras).
2. La **IoU** mide cuánto se solapan dos cajas o máscaras; una detección cuenta como correcta si IoU ≥ 0.5.
3. Los detectores producen muchas cajas repetidas; **NMS** se queda con la más segura de cada grupo.
4. La **U-Net** baja para entender y sube para dibujar, con atajos que conservan el detalle; la segmentación se evalúa con **IoU/Dice**, no con exactitud de píxeles.
5. Un **Vision Transformer** trata la imagen como una frase de **parches**; necesita más datos que una CNN, pero preentrenado a gran escala es el estado del arte y los "ojos" de los modelos multimodales.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** Dos cajas idénticas tienen IoU…
- a) 0 · b) 0.5 · c) 1 · d) Depende de su tamaño

**2.** Un detector muestra tres cajas casi iguales sobre el mismo perro. ¿Qué falta o está mal ajustado?
- a) El umbral de confianza · b) La NMS · c) La resolución · d) El tamaño de parche

**3.** Quieres medir el área exacta de cada lesión en fotos de piel, aunque haya varias lesiones juntas. Necesitas…
- a) Clasificación · b) Detección · c) Segmentación semántica · d) Segmentación de instancias

**4.** Imagen de 224 × 224, parches de 14 × 14. ¿Cuántos parches ve el ViT?
- a) 16 · b) 196 · c) 256 · d) 3,136

**5.** ¿Por qué es mala idea evaluar una segmentación de tumores con exactitud de píxeles?
- a) Porque es lenta de calcular
- b) Porque el fondo domina y un modelo que no detecta nada saca una exactitud altísima
- c) Porque solo sirve para detección
- d) Porque la U-Net no da probabilidades

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** La intersección es igual a la unión.
2. **b)** NMS elimina las cajas repetidas.
3. **d)** Necesitas separar cada lesión (instancia) y su forma exacta.
4. **c)** 224 / 14 = 16 → 16 × 16 = 256.
5. **b)** Mira el ejercicio 10: 96 % de exactitud con 0 % de IoU.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** El CAM de una red que clasifica "lobo" vs. "husky" se ilumina sobre todo en la **nieve** del fondo, no en el animal. ¿Qué te dice eso del modelo y de los datos? ¿Qué harías?

**B. Caso 💼** Una cadena de supermercados quiere detectar estantes vacíos con cámaras. Dos errores posibles: (1) avisar de un estante vacío que no lo está (un empleado camina en vano) y (2) no avisar de un estante vacío (se pierden ventas). La gerencia dice que el error 2 es **mucho** más caro. ¿Subirías o bajarías el umbral de confianza del detector? ¿Qué métrica vigilarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El modelo aprendió un **atajo espurio**: en los datos de entrenamiento casi todos los lobos aparecían en la nieve y los huskies no (o al revés), así que "nieve = lobo" funcionaba. Fallará con un husky en la nieve. Hay que **diversificar los datos** (lobos sin nieve, huskies con nieve), quizás recortar el fondo o usar *augmentation*, y volver a revisar con CAM. Es un ejemplo clásico de por qué la **explicabilidad** importa (Lección 15.6).

**B.** **Bajarlo**: aceptar más falsas alarmas a cambio de escapar menos estantes vacíos. La métrica a vigilar es el **recall** de "estante vacío" (sin dejar que la precisión caiga tanto que los empleados ignoren las alertas). Igual que en la Lección 4.8: el umbral se elige por el **costo** de cada error, no por defecto.
</details>

### 10.3 🧩 Práctica integradora: "Evalúa un detector de autos"

Una empresa de estacionamientos probó un detector en una foto. Tienes las **cajas reales** (dibujadas por una persona) y las **predicciones** del detector con su confianza. Vas a evaluarlo como se hace en la industria.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
autos_reales = np.array([[10, 10, 60, 40], [70, 10, 120, 40], [130, 10, 180, 40], [10, 60, 60, 90], [70, 60, 120, 90]])
predicciones = np.array([
    [12, 12, 62, 42],     # p0
    [11,  9, 59, 41],     # p1
    [72, 12, 118, 38],    # p2
    [142, 15, 192, 45],   # p3
    [10, 62, 58, 92],     # p4
    [150, 60, 190, 90],   # p5
])
conf_pred = np.array([0.95, 0.80, 0.90, 0.60, 0.85, 0.30])

fig, ax = plt.subplots(figsize=(8, 4)); ax.set_xlim(0, 200); ax.set_ylim(100, 0); ax.set_aspect("equal")
for c in autos_reales:
    ax.add_patch(patches.Rectangle(c[:2], c[2] - c[0], c[3] - c[1], fill=True, alpha=0.25, color="green"))
for k, (c, s) in enumerate(zip(predicciones, conf_pred)):
    ax.add_patch(patches.Rectangle(c[:2], c[2] - c[0], c[3] - c[1], fill=False, color="red", lw=1.5))
    ax.text(c[0], c[3] + 4, f"p{k}:{s:.2f}", color="red", fontsize=8)
ax.set_title("Verde = autos reales · Rojo = predicciones"); plt.show()

**Paso 2 🔮 — Aplica NMS.** Antes de ejecutar: ¿qué predicción se eliminará? (Mira p0 y p1.)

In [ ]:
quedan = nms_a_mano(predicciones, conf_pred, umbral=0.5)
print("Predicciones que sobreviven a NMS:", [f"p{i}" for i in quedan])

<details><summary><b>👀 Ver respuesta</b></summary>

Se elimina **p1**: se solapa muchísimo con p0 (el mismo auto) y tiene menos confianza (0.80 < 0.95). Sobreviven p0, p2, p4, p3 y p5 (ordenadas por confianza).
</details>

**Paso 3 ✋ — Empareja predicciones y autos.** Una predicción es **acierto** (verdadero positivo) si tiene IoU ≥ 0.5 con un auto real que todavía no fue emparejado. Si no, es **falsa alarma**. Los autos reales que se quedan sin pareja son **autos no detectados**.

Antes de ejecutar, mirando el dibujo: ¿qué predicciones crees que serán falsas alarmas? ¿Qué auto se quedará sin detectar?

In [ ]:
def evaluar(quedan, umbral_conf=0.0, umbral_iou=0.5):
    emparejados, aciertos, falsas = set(), 0, 0
    for i in quedan:                                    # ya vienen ordenadas por confianza
        if conf_pred[i] < umbral_conf:
            continue
        ious = [iou(predicciones[i], r) for r in autos_reales]
        mejor = int(np.argmax(ious))
        if ious[mejor] >= umbral_iou and mejor not in emparejados:
            emparejados.add(mejor); aciertos += 1
            print(f"  p{i} ✅ acierto con el auto {mejor} (IoU {ious[mejor]:.2f})")
        else:
            falsas += 1
            print(f"  p{i} ❌ falsa alarma (mejor IoU {ious[mejor]:.2f})")
    no_detectados = len(autos_reales) - aciertos
    precision = aciertos / max(aciertos + falsas, 1)
    recall = aciertos / len(autos_reales)
    print(f"  → aciertos {aciertos} · falsas alarmas {falsas} · no detectados {no_detectados} · "
          f"precisión {precision:.2f} · recall {recall:.2f}")
    return precision, recall

evaluar(quedan)

<details><summary><b>👀 Ver respuesta</b></summary>

- p0, p2 y p4 son **aciertos** (IoU alta con los autos 0, 1 y 3).
- **p3** es una **falsa alarma** aunque está "cerca" del auto 2: está desplazada 12 píxeles en x y 5 en y, y su IoU (≈0.46) no llega a 0.5.
- **p5** es una falsa alarma: no hay ningún auto ahí (IoU 0).
- Quedan **sin detectar** el auto 2 (la caja de p3 no alcanzó) y el auto 4 (nadie lo predijo).

Precisión 3/5 = **0.60**, recall 3/5 = **0.60**.
</details>

**Paso 4 ✏️ — Mueve el umbral de confianza.** Prueba `umbral_conf` = 0.5 y 0.9. ¿Qué pasa con la precisión y el recall?

In [ ]:
for umbral in [0.0, 0.5, 0.9]:                 # ✏️ prueba otros valores
    print(f"Umbral de confianza = {umbral}")
    evaluar(quedan, umbral_conf=umbral)

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.0**: precisión 0.60 · recall 0.60.
- **0.5**: desaparece p5 (0.30) → precisión 3/4 = **0.75**, recall igual **0.60**.
- **0.9**: solo quedan p0 (0.95) y p2 (0.90) → precisión **1.00**, recall 2/5 = **0.40**.

Subir el umbral sube la precisión y baja el recall: la curva precisión–recall de la Lección 4.3. El **mAP** es, en esencia, el área bajo esa curva, promediada sobre todas las clases.
</details>

**Paso 5 ✋ — Tu informe.** En tres frases para el gerente del estacionamiento (que no sabe ML): ¿qué tan bueno es el detector y qué le pedirías al equipo?

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algo como: *"En esta foto, el detector encontró 3 de los 5 autos y dio 2 avisos falsos. Uno de los autos que 'no encontró' en realidad lo vio, pero dibujó la caja algo desplazada, así que se puede mejorar ajustando la precisión de las cajas; otro auto lo pasó por alto por completo. Antes de decidir, hay que evaluarlo con cientos de fotos (de día, de noche, con lluvia), no con una sola."*
</details>

---
## 📚 Para profundizar
- 📘 **Artículos clave:** Redmon et al. (2016), *You Only Look Once* (YOLO) · Ronneberger et al. (2015), *U-Net* · Dosovitskiy et al. (2020), *An Image is Worth 16×16 Words* (ViT) · Kirillov et al. (2023), *Segment Anything*.
- 🎮 **Interactivo:** demo en línea de *Segment Anything* (segment-anything.com): segmenta cualquier objeto con un clic.
- 📘 **Documentación:** `torchvision.models.detection` y `torchvision.ops` (IoU, NMS, conversión de cajas); Ultralytics YOLO para detección práctica (para Colab).
- 📺 **Video:** CS231n de Stanford, clase "Detection and Segmentation" (inglés).

---
⬅️ **Anterior:** [10.5 Transfer learning y fine-tuning: no empezar de cero](10.5-transfer-learning.ipynb) · ➡️ **Siguiente:** [10.7 Proyecto del módulo: Clasificador de vehículos para una empresa de logística 🚚✈️🚢](10.7-proyecto-del-modulo.md)